In [1]:
import torch
import sys
import json

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Python: {sys.version}")

PyTorch: 2.9.0+cu126
CUDA: 12.6
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install vllm

INFO: pip is looking at multiple versions of model-hosting-container-standards to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.3/370.3 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import vllm
print(vllm.__version__)

0.11.2


In [4]:
from vllm import LLM, SamplingParams
import os

In [ ]:
entries[0]

{'qid': 2880,
 'original_prompt': 'How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems?',
 'generated_prompt': 'Discuss the evolutionary significance of bioluminescence in marine organisms such as jellyfish, octopuses, and squid. Provide examples of how bioluminescence has influenced the behavior and survival of these species over time. Use real-world evidence to support your arguments and consider the ecological implications of bioluminescent behaviors on the overall health and diversity of marine ecosystems.'}

# Qwen Judge

In [60]:
from vllm import LLM, SamplingParams
import os

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
INPUT_FILE = "/content/drive/MyDrive/AdvNLP/generated_prompts_post_finetune_output.jsonl"

In [9]:
from vllm import LLM, SamplingParams

# --- LLM Initialization ---
llm = LLM(
    model=MODEL_NAME,
    max_model_len=8192,
    gpu_memory_utilization=0.92,
    tensor_parallel_size=1,
    dtype="bfloat16"
)

# --- Sampling Parameters ---
sampling_params = SamplingParams(
    temperature=0.1,
    top_p=0.95,
    max_tokens=1024,
    stop_token_ids=[llm.get_tokenizer().eos_token_id]
)

print("Qwen2.5-7B-Instruct loaded and ready!")

INFO 11-27 05:58:17 [utils.py:253] non-default args: {'dtype': 'bfloat16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.92, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

INFO 11-27 05:58:39 [model.py:631] Resolved architecture: Qwen2ForCausalLM
INFO 11-27 05:58:39 [model.py:1745] Using max model len 8192
INFO 11-27 05:58:42 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

WARNING 11-27 05:58:45 [system_utils.py:103] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 11-27 06:00:53 [llm.py:352] Supported tasks: ['generate']
Qwen2.5-7B-Instruct loaded and ready!


In [61]:
import json


data = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"Loaded {len(data)} entries for judgment")

Loaded 552 entries for judgment


In [7]:
JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator of LLM responses.

You will be given:
- A text input
- Two different answers generated by two different prompting methods

Your task:
Evaluate which answer is objectively better for a human user.

Judge ONLY the answers, NOT the prompts used to produce them.

Evaluation criteria:
- Accuracy and correctness
- Completeness and relevance
- Clarity and readability
- Usefulness for a human user
- Helpful level of detail
- Avoiding unnecessary repetition or confusion

STRICT RULES:
- You MUST NOT reference, consider, or evaluate any prompt wording.
- Judge solely the final answers and how well they serve the human user.
- Ignore answer length differences unless one is clearly less useful.
- Do not reveal chain-of-thought or step-by-step reasoning.

Return ONLY valid JSON in this format:

{
  "winner": "A" | "B" ,
  "A_score": 1-10,
  "B_score": 1-10,
  "feedback": "2-3 concise sentences explaining the choice"
}
"""
JUDGE_USER_TEMPLATE = """
Input:
{original_prompt}

Answer A:
{original_response}

Answer B:
{generated_response}

Compare Answer A and Answer B STRICTLY based on answer quality alone.

Return ONLY the JSON specified in the system instructions.
"""


In [63]:
# Build full Qwen2.5 chat format for every entry in one go
prompts = []
mapping = []  # remembers which data index each prompt belongs to

for idx, entry in enumerate(data):
    user_msg = JUDGE_USER_TEMPLATE.format(
        original_prompt=entry["original_prompt"],
        original_response=entry["original_prompt_response"],
        # generated_prompt=entry["generated"]["prompt"],
        generated_response=entry["generated_prompt_response"]
    )

    full_prompt = (
        f"<|im_start|>system\n{JUDGE_SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    prompts.append(full_prompt)
    mapping.append(idx)

print(f"Prepared {len(prompts)} prompts → ready for single-shot inference!")

Prepared 552 prompts → ready for single-shot inference!


In [64]:
print("Running single batch inference on Qwen2.5-7B-Instruct...")
outputs = llm.generate(prompts, sampling_params, use_tqdm=True)

print(f"Batch complete! Got {len(outputs)} judgments.")

Running single batch inference on Qwen2.5-7B-Instruct...


Adding requests:   0%|          | 0/552 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/552 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batch complete! Got 552 judgments.


In [15]:
import re
import json

def safe_extract_json(text):
    """
    Robustly extracts the first valid JSON object from LLM output.
    Handles cases where model adds text before/after the {} block.
    """
    if not text:
        return {"winner": "error", "original_score": 0, "generated_score": 0, "feedback": "Empty response"}

    # Method 1: Look for a block that contains "winner" (most reliable)
    match = re.search(r'\{[^}]*"winner"[^}]*\}', text, re.DOTALL)
    if not match:
        # Method 2: Grab the first { ... } block
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            # Find matching closing brace
            brace_count = 0
            for i, char in enumerate(match.group(0)):
                if char == '{':
                    brace_count += 1
                elif char == '}':
                    brace_count -= 1
                    if brace_count == 0:
                        json_candidate = match.group(0)[:i+1]
                        break
            else:
                json_candidate = match.group(0)
        else:
            json_candidate = ""
    else:
        json_candidate = match.group(0)

    if not json_candidate.strip():
        return {"winner": "error", "A_score": 0, "B_score": 0, "feedback": "No JSON found"}

    try:
        parsed = json.loads(json_candidate)
        # Basic validation
        if "winner" in parsed and "A_score" in parsed and "B_score" in parsed:
            return parsed
        else:
            return {"winner": "error", "A_score": 0, "B_score": 0, "feedback": "Missing required keys"}
    except json.JSONDecodeError:
        return {"winner": "error", "A_score": 0, "B_score": 0, "feedback": f"Invalid JSON: {str(json_candidate)[:100]}..."}

In [65]:
# Attach judgments
for output, data_idx in zip(outputs, mapping):
    judgment = safe_extract_json(output.outputs[0].text)
    data[data_idx]["judgment"] = judgment

In [66]:
# Save final file
OUTPUT_FILE = "/content/drive/MyDrive/AdvNLP/NLI_Post1_WITH_QWEN_JUDGMENTAB.jsonl"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved {len(data)} entries → {OUTPUT_FILE}")

Saved 552 entries → /content/drive/MyDrive/AdvNLP/NLI_Post1_WITH_QWEN_JUDGMENTAB.jsonl


In [67]:
# Instant results
valid = [x for x in data if x["judgment"].get("winner") in ["A", "B", "tie"]]
orig_wins = sum(1 for x in valid if x["judgment"]["winner"] == "A")
gen_wins  = sum(1 for x in valid if x["judgment"]["winner"] == "B")
ties      = len(valid) - orig_wins - gen_wins

print(f"\nJUDGMENT RESULTS ({len(valid)} valid):")
print(f"Original wins : {orig_wins:3d} ({orig_wins/len(valid)*100:5.1f}%)")
print(f"Improved wins : {gen_wins:3d} ({gen_wins/len(valid)*100:5.1f}%)")
# print(f"Ties          : {ties:3d} ({ties/len(valid)*100:5.1f}%)")
print(f"Avg score — Original : {sum(x['judgment'].get('A_score',0) for x in valid)/len(valid):.2f}/10")
print(f"Avg score — Improved : {sum(x['judgment'].get('B_score',0) for x in valid)/len(valid):.2f}/10")


JUDGMENT RESULTS (545 valid):
Original wins : 394 ( 72.3%)
Improved wins : 151 ( 27.7%)
Avg score — Original : 7.23/10
Avg score — Improved : 5.54/10


In [59]:
print("Pre Score")
# Instant results
valid = [x for x in data if x["judgment"].get("winner") in ["A", "B", "tie"]]
orig_wins = sum(1 for x in valid if x["judgment"]["winner"] == "A")
gen_wins  = sum(1 for x in valid if x["judgment"]["winner"] == "B")
ties      = len(valid) - orig_wins - gen_wins

print(f"\nJUDGMENT RESULTS ({len(valid)} valid):")
print(f"Original wins : {orig_wins:3d} ({orig_wins/len(valid)*100:5.1f}%)")
print(f"Improved wins : {gen_wins:3d} ({gen_wins/len(valid)*100:5.1f}%)")
print(f"Ties          : {ties:3d} ({ties/len(valid)*100:5.1f}%)")
print(f"Avg score — Original : {sum(x['judgment'].get('A_score',0) for x in valid)/len(valid):.2f}/10")
print(f"Avg score — Improved : {sum(x['judgment'].get('B_score',0) for x in valid)/len(valid):.2f}/10")

Pre Score

JUDGMENT RESULTS (547 valid):
Original wins : 383 ( 70.0%)
Improved wins : 164 ( 30.0%)
Ties          :   0 (  0.0%)
Avg score — Original : 7.23/10
Avg score — Improved : 5.73/10


In [ ]:
# LLAMA Judge

In [6]:
import gc
import torch
from vllm import LLM, SamplingParams

# Cleanup any previous vLLM instance to free up GPU memory
if 'llm' in globals():
    print("Clearing previous LLM instance...")
    del llm
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("Previous LLM instance cleared.")

# CELL 1 — Load model (your code is perfect)

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

judge_llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    max_model_len=8192,
    # gpu_memory_utilization=0.92,
    tensor_parallel_size=1,
    trust_remote_code=True
)

judge_params = SamplingParams(
    temperature=0.0,
    max_tokens=512,
    stop=["<|eot_id|>", "<|end_of_text|>"],
)

print("DeepSeek-R1-Distill-Llama-8B loaded")

INFO 11-27 06:42:03 [utils.py:253] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 8192, 'gpu_memory_utilization': 0.92, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

INFO 11-27 06:42:21 [model.py:631] Resolved architecture: LlamaForCausalLM
INFO 11-27 06:42:21 [model.py:1745] Using max model len 8192
INFO 11-27 06:42:24 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

WARNING 11-27 06:42:28 [system_utils.py:103] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 11-27 06:44:37 [llm.py:352] Supported tasks: ['generate']
DeepSeek-R1-Distill-Llama-8B loaded


In [15]:
import json
INPUT_FILE = "/content/drive/MyDrive/AdvNLP/generated_prompts_pre_finetune_output.jsonl"
# Load your file
with open(INPUT_FILE) as f:
    data = [json.loads(line) for line in f if line.strip()]


In [16]:
prompts = []
idx_map = []

for i, item in enumerate(data):
    # This will work — we doubled the braces
    user_msg = JUDGE_USER_TEMPLATE.format(
        original_prompt=item["original_prompt"],
        original_response=item["original_prompt_response"],
        # generated_prompt=entry["generated"]["prompt"],
        generated_response=item["generated_prompt_response"]
    )

    # Correct DeepSeek-R1 chat template + we start the JSON ourselves
    prompt = (
        "<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id>\n\n{JUDGE_SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id>\n\n"
        f"{user_msg}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id>\n\n"
        '{"winner": "'     # ← model continues → always valid JSON
    )

    prompts.append(prompt)
    idx_map.append(i)

print(f"Prepared {len(prompts)} prompts — NO MORE KEYERROR")

Prepared 552 prompts — NO MORE KEYERROR


In [22]:
print("Sample prompt:", prompts[3] + "...")

Sample prompt: <|begin_of_text|><|start_header_id|>system<|end_header_id>


You are an expert evaluator of LLM responses.

You will be given:
- A text input
- Two different answers generated by two different prompting methods

Your task:
Evaluate which answer is objectively better for a human user.

Judge ONLY the answers, NOT the prompts used to produce them.

Evaluation criteria:
- Accuracy and correctness
- Completeness and relevance
- Clarity and readability
- Usefulness for a human user
- Helpful level of detail
- Avoiding unnecessary repetition or confusion

STRICT RULES:
- You MUST NOT reference, consider, or evaluate any prompt wording.
- Judge solely the final answers and how well they serve the human user.
- Ignore answer length differences unless one is clearly less useful.
- Do not reveal chain-of-thought or step-by-step reasoning.

Return ONLY valid JSON in this format:

{
  "winner": "A" | "B" ,
  "A_score": 1-10,
  "B_score": 1-10,
  "feedback": "2-3 concise sentences ex

In [26]:
# CELL 3 — Run the batch
print("Running judgment batch...")
outputs = judge_llm.generate(prompts, judge_params, use_tqdm=True)
print("Finished!")

Running judgment batch...


Adding requests:   0%|          | 0/552 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/552 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Finished!


In [14]:
import json

def extract_json(text):
    try:
        s = text.find("{")
        e = text.rfind("}") + 1
        return json.loads(text[s:e])
    except:
        return {"winner":"error","A_score":0,"B_score":0,"feedback":"parse failed"}

for out, i in zip(outputs, idx_map):
    data[i]["judgment"] = extract_json(out.outputs[0].text)

OUT = "/content/drive/MyDrive/AdvNLP/NI_post_deepseek_eval.jsonl"
with open(OUT, "w", encoding="utf-8") as f:
    for x in data:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

valid = [x for x in data if x["judgment"]["winner"] in ["A","B","tie"]]
orig = sum(1 for x in valid if x["judgment"]["winner"]=="A")
gen  = sum(1 for x in valid if x["judgment"]["winner"]=="B")
tie  = len(valid) - orig - gen

print(f"\nFINAL RESULTS POST({len(valid)}/{len(data)} perfect judgments):")
print(f"Original wins : {orig:3d} ({orig/len(valid)*100:5.1f}%)")
print(f"Improved wins : {gen:3d} ({gen/len(valid)*100:5.1f}%)")
# print(f"Ties          : {tie:3d} ({tie/len(valid)*100:5.1f}%)")
print(f"Avg Original  : {sum(x['judgment']['A_score'] for x in valid)/len(valid):.2f}/10")
print(f"Avg Improved  : {sum(x['judgment']['B_score'] for x in valid)/len(valid):.2f}/10")
print(f"\nSaved → {OUT}")


FINAL RESULTS POST(551/552 perfect judgments):
Original wins : 176 ( 31.9%)
Improved wins : 375 ( 68.1%)
Avg Original  : 6.12/10
Avg Improved  : 7.56/10

Saved → /content/drive/MyDrive/AdvNLP/NI_post_deepseek_eval.jsonl


In [27]:
# CELL 4 — Extract + save + real results (no more zero valid)
import json

def extract_json(text):
    try:
        s = text.find("{")
        e = text.rfind("}") + 1
        return json.loads(text[s:e])
    except:
        return {"winner":"error","A_score":0,"B_score":0,"feedback":"parse failed"}

for out, i in zip(outputs, idx_map):
    data[i]["judgment"] = extract_json(out.outputs[0].text)

OUT = "/content/drive/MyDrive/AdvNLP/NI_pre_deepseek_eval.jsonl"
with open(OUT, "w", encoding="utf-8") as f:
    for x in data:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

valid = [x for x in data if x["judgment"]["winner"] in ["A","B","tie"]]
orig = sum(1 for x in valid if x["judgment"]["winner"]=="A")
gen  = sum(1 for x in valid if x["judgment"]["winner"]=="B")
tie  = len(valid) - orig - gen

print(f"\nFINAL RESULTS Pre({len(valid)}/{len(data)} perfect judgments):")
print(f"Original wins : {orig:3d} ({orig/len(valid)*100:5.1f}%)")
print(f"Improved wins : {gen:3d} ({gen/len(valid)*100:5.1f}%)")
# print(f"Ties          : {tie:3d} ({tie/len(valid)*100:5.1f}%)")
print(f"Avg Original  : {sum(x['judgment']['A_score'] for x in valid)/len(valid):.2f}/10")
print(f"Avg Improved  : {sum(x['judgment']['B_score'] for x in valid)/len(valid):.2f}/10")
print(f"\nSaved → {OUT}")


FINAL RESULTS Pre(552/552 perfect judgments):
Original wins : 177 ( 32.1%)
Improved wins : 375 ( 67.9%)
Avg Original  : 6.16/10
Avg Improved  : 7.61/10

Saved → /content/drive/MyDrive/AdvNLP/NI_pre_deepseek_eval.jsonl
